In [1]:
#| default_exp models.ml_multi_forecaster

In [2]:
#| export
from __future__ import annotations
from typing import List, Dict, Optional, Callable, Tuple, Any, Union
import numpy as np
import pandas as pd
import copy
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from peshbeen.model_selection import SplitTimeSeries
from peshbeen.statstools import lr_trend_model, forecast_trend
from peshbeen.transformations import (
    box_cox_transform, back_box_cox_transform,
    rolling_quantile, expanding_mean, expanding_std, expanding_quantile
)
from peshbeen.helpers import seasonal_diff, undiff_ts, invert_seasonal_diff
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings("ignore")

class ml_multi_forecaster:
    """
    Multi-Series Interdependent Machine Learning Forecaster (ml_multi_forecaster).

    Forecasts multiple time series simultaneously using long-format panel data.
    It automatically constructs a feature matrix that includes historical lag values
    and transformations of ALL series to capture cross-series interdependence,
    supporting target variable scaling (StandardScaler/RobustScaler),
    categorical feature encoding, exogenous variables, and multi-fold cross-validation.
    """
    def __init__(
        self,
        model: Any,
        id_col: str,
        target_col: str,
        lags: Optional[Union[int, List[int], Dict[str, Union[int, List[int]]]]] = None,
        lag_transform: Optional[Union[list, Dict[str, list]]] = None,
        series_encoding: Optional[str] = 'dummy',
        difference: Optional[Union[int, Dict[str, int]]] = None,
        seasonal_diff: Optional[Union[int, Dict[str, int]]] = None,
        trend: Optional[Union[str, Dict[str, str]]] = None,
        pol_degree: Union[int, Dict[str, int]] = 1,
        ets_params: Optional[Dict[str, Any]] = None,
        change_points: Optional[Union[List[int], Dict[str, List[int]]]] = None,
        box_cox: Union[bool, float, int, Dict[str, Union[bool, float, int]]] = False,
        box_cox_biasadj: Union[bool, Dict[str, bool]] = False,
        target_scaler: Optional[Any] = None,
        cat_variables: Optional[List[str]] = None,
        categorical_encoder: Optional[Any] = None
    ) -> None:
        """
        Initialize the ml_multi_forecaster.

        Parameters
        ----------
        model : Any
            A scikit-learn regressor model object (e.g. LGBMRegressor(), Ridge(), CatBoostRegressor(), LinearRegression(), etc.).
        id_col : str
            Column name identifying each time series (e.g. 'store_item' or 'item_id').
        target_col : str
            Column name of the target numerical values (e.g. 'sales' or 'flow').
        lags : int, list of int, or dict, optional
            Lags to include as features. If an integer, lags 1..lags will be generated for all series.
            If a list of ints, those specific lags will be generated for all series.
            If a dictionary keyed by series ID, customized lags can be provided per series.
        lag_transform : list or dict, optional
            List of lag-transform functions applied to all series, or dict mapping series ID to list of functions.
        series_encoding : str, optional
            Strategy to encode series identifier column. Options are 'dummy' (one-hot), 'ordinal' (label integer),
            or None (native categorical feature handling for LightGBM/CatBoost). Default is 'dummy'.
        difference : int or dict, optional
            Order of ordinary differencing per series or single int for all series.
        seasonal_diff : int or dict, optional
            Seasonal period for seasonal differencing per series or single int for all series.
        trend : str or dict, optional
            Trend strategy: 'linear' or 'ets'.
        pol_degree : int or dict, optional
            Polynomial degree for linear trend removal (default is 1).
        ets_params : dict, optional
            Parameters for ExponentialSmoothing when trend='ets'.
        change_points : list or dict, optional
            List of breakpoint indices for piecewise linear trend fitting.
        box_cox : bool, float, int, or dict, optional
            Box-Cox transformation setting per series or globally.
        box_cox_biasadj : bool or dict, optional
            Whether to apply bias adjustment on Box-Cox back-transformation.
        target_scaler : object or dict, optional
            Scaler object from sklearn.preprocessing (e.g. StandardScaler(), RobustScaler(), MinMaxScaler())
            to scale target variables after differencing/detrending, before feature extraction.
            Can be a single scaler applied to all series or a dict mapping series ID to a scaler.
        cat_variables : list of str, optional
            List of categorical feature column names (e.g. ['day_of_week', 'month']).
        categorical_encoder : object, optional
            Categorical encoder object (e.g. OneHotEncoder(sparse_output=False)).
        """
        self.model = model
        self.model_name = self.model.__class__.__name__
        self.id_col = id_col
        self.target_col = target_col
        self.series_encoding = series_encoding

        # Validate series_encoding strategy against supported model capabilities
        if self.series_encoding is None:
            if self.model_name not in ["LGBMRegressor", "CatBoostRegressor"]:
                raise ValueError(
                    "series_encoding=None is only supported for LGBMRegressor and CatBoostRegressor. "
                    "Please set series_encoding='dummy' or 'ordinal'."
                )

        self.lags = lags
        self.lag_transform = lag_transform
        self.difference = difference
        self.seasonal_diff = seasonal_diff
        self.trend = trend
        self.pol_degree = pol_degree
        self.ets_params = ets_params or {}
        self.change_points = change_points
        self.box_cox = box_cox
        self.box_cox_biasadj = box_cox_biasadj
        self.target_scaler = target_scaler
        self.cat_variables = cat_variables
        self.cat_encoder = categorical_encoder
        self.cat_dtypes = {}

        # Validate categorical variable handling when no encoder is specified
        if self.cat_variables is not None and self.cat_encoder is None:
            if self.model_name not in ["LGBMRegressor", "CatBoostRegressor"]:
                raise ValueError(
                    "Model must be LGBMRegressor or CatBoostRegressor to handle categorical variables without a specified encoder "
                    "(or provide an encoder such as OneHotEncoder)."
                )

    def _get_per_series_param(self, param: Any, series_id: str, default: Any = None) -> Any:
        """Helper to retrieve per-series customized hyperparameter or global default."""
        if isinstance(param, dict):
            return param.get(series_id, default)
        return param if param is not None else default

    def _normalize_lags(self, series_ids: List[str]) -> Dict[str, List[int]]:
        """Normalize lag specifications (int, list, or dict) into explicit lag integer lists per series."""
        result = {}
        for s in series_ids:
            if isinstance(self.lags, dict):
                v = self.lags.get(s, None)
            else:
                v = self.lags
            
            if v is None:
                result[s] = []
            elif isinstance(v, int):
                result[s] = list(range(1, v + 1))
            elif isinstance(v, list):
                result[s] = v
            else:
                raise TypeError(f"Lags for series '{s}' must be int, list of ints, or None.")
        return result

    def create_encoded_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Align categorical feature dtypes and encode categorical features using ColumnTransformer.
        """
        dfc = df.copy()
        if self.cat_variables is not None:
            # 1. Enforce consistent categorical categories and ordering across train and forecast
            for col in self.cat_variables:
                if col in dfc.columns:
                    if col not in self.cat_dtypes:
                        cats = sorted(dfc[col].dropna().unique().tolist())
                        self.cat_dtypes[col] = pd.CategoricalDtype(categories=cats)
                    dfc[col] = pd.Categorical(dfc[col], dtype=self.cat_dtypes[col])
            
            # 2. Encode using fitted ColumnTransformer if categorical_encoder is supplied
            if self.cat_encoder is not None:
                if self.target_col in dfc.columns:
                    num_cols = [c for c in dfc.columns if c not in self.cat_variables + [self.target_col, self.id_col]]
                    self.preprocess = ColumnTransformer(
                        transformers=[("cat", self.cat_encoder, self.cat_variables), ("num", "passthrough", num_cols)],
                        remainder="drop",
                        verbose_feature_names_out=False
                    ).set_output(transform="pandas")
                    
                    target_series = dfc[self.target_col]
                    X_encoded = self.preprocess.fit_transform(dfc.drop(columns=[self.target_col, self.id_col]), y=target_series)
                    return pd.concat([dfc[[self.id_col, self.target_col]], X_encoded], axis=1)
                else:
                    id_series = dfc[self.id_col] if self.id_col in dfc.columns else None
                    X_drop = dfc.drop(columns=[self.id_col]) if self.id_col in dfc.columns else dfc
                    X_encoded = self.preprocess.transform(X_drop)
                    if id_series is not None:
                        return pd.concat([dfc[[self.id_col]], X_encoded], axis=1)
                    return X_encoded
        return dfc

    def data_prep(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
        """
        Prepare the panel feature matrix across all series.

        Steps:
        1. Encode categorical features.
        2. Pivot long-format DataFrame into a wide matrix (time_index x series_ids).
        3. Apply per-series transformations in order: Box-Cox -> Trend Removal -> Ordinary Diff -> Seasonal Diff -> Target Scaling.
        4. Construct cross-series lag feature matrix using historical values of ALL series.
        5. Stack wide features into a panel training matrix (X, y) with preserved time index.
        """
        dfc = df.copy()
        
        # 1. Encode categorical feature columns
        if self.cat_variables is not None:
            dfc = self.create_encoded_features(dfc)

        series_ids = sorted(dfc[self.id_col].unique().tolist())
        self.series_ids = series_ids
        self.cat_type = pd.CategoricalDtype(categories=series_ids)

        exog_cols = [c for c in dfc.columns if c not in [self.id_col, self.target_col]]
        self.exog_cols = exog_cols

        # 2. Pivot long format to wide matrix (rows = time steps, cols = series identifiers)
        wide_orig = dfc.pivot(columns=self.id_col, values=self.target_col)
        self.wide_orig = wide_orig.copy()
        wide_trans = wide_orig.copy()
        
        self.transform_meta = {s: {} for s in series_ids}
        normalized_lags = self._normalize_lags(series_ids)
        self.normalized_lags = normalized_lags

        # 3. Apply per-series forward transformations
        for s in series_ids:
            meta = self.transform_meta[s]
            meta['orig_series'] = wide_orig[s].copy()
            s_data = wide_trans[s].copy()

            # Forward Step A: Box-Cox Transformation (Variance stabilization)
            bc_param = self._get_per_series_param(self.box_cox, s, False)
            if bc_param:
                lmda = None if isinstance(bc_param, bool) else bc_param
                is_zero = np.any(s_data.dropna() < 1)
                trans_s, final_lmda = box_cox_transform(x=s_data, shift=is_zero, box_cox_lmda=lmda)
                meta['box_cox'] = True
                meta['is_zero'] = is_zero
                meta['lmda'] = final_lmda
                meta['biasadj'] = self._get_per_series_param(self.box_cox_biasadj, s, False)
                s_data = pd.Series(trans_s, index=s_data.index)
            else:
                meta['box_cox'] = False

            # Forward Step B: Trend Removal (Linear polynomial / Piecewise or ETS)
            tr_param = self._get_per_series_param(self.trend, s, None)
            if tr_param is not None:
                meta['trend_type'] = tr_param
                pol = self._get_per_series_param(self.pol_degree, s, 1)
                cps = self._get_per_series_param(self.change_points, s, None)
                if tr_param == 'linear':
                    if cps is not None:
                        trend_vals, lr_mod, X_tr = lr_trend_model(s_data, degree=pol, breakpoints=cps, type='piecewise')
                    else:
                        trend_vals, lr_mod, X_tr = lr_trend_model(s_data, degree=pol)
                    meta['lr_model'] = lr_mod
                    meta['pol_degree'] = pol
                    meta['cps'] = cps
                    meta['trend_vals'] = trend_vals
                    s_data = s_data - trend_vals
                elif tr_param == 'ets':
                    ets_p = self.ets_params or {}
                    ets_m = ExponentialSmoothing(s_data, **{k: v for k, v in ets_p.items() if k in ["trend","damped_trend", "seasonal","seasonal_periods"]}).fit()
                    meta['ets_model_fit'] = ets_m
                    meta['trend_vals'] = ets_m.fittedvalues.values
                    s_data = s_data - meta['trend_vals']
            else:
                meta['trend_type'] = None

            # Forward Step C: Ordinary Differencing
            diff_param = self._get_per_series_param(self.difference, s, None)
            if diff_param is not None:
                meta['difference'] = diff_param
                meta['orig_before_diff'] = s_data.tolist()
                s_data = pd.Series(
                    np.diff(s_data, n=diff_param, prepend=np.repeat(np.nan, diff_param)),
                    index=s_data.index
                )
            else:
                meta['difference'] = None

            # Forward Step D: Seasonal Differencing
            sdiff_param = self._get_per_series_param(self.seasonal_diff, s, None)
            if sdiff_param is not None:
                meta['seasonal_diff'] = sdiff_param
                meta['orig_before_sdiff'] = s_data.tolist()
                s_data = pd.Series(seasonal_diff(s_data, sdiff_param), index=s_data.index)
            else:
                meta['seasonal_diff'] = None

            # Forward Step E: Target Scaling (StandardScaler/RobustScaler after diff, before feature extraction)
            scaler_param = self._get_per_series_param(self.target_scaler, s, None)
            if scaler_param is not None:
                scaler_inst = clone(scaler_param) if hasattr(scaler_param, 'fit') else clone(scaler_param)
                s_vals = s_data.values.reshape(-1, 1)
                valid_mask_s = ~np.isnan(s_vals.ravel())
                if np.any(valid_mask_s):
                    scaler_inst.fit(s_vals[valid_mask_s].reshape(-1, 1))
                    s_scaled = s_data.copy()
                    s_scaled.iloc[valid_mask_s] = scaler_inst.transform(s_vals[valid_mask_s].reshape(-1, 1)).ravel()
                    s_data = s_scaled
                    meta['scaler'] = scaler_inst
                else:
                    meta['scaler'] = None
            else:
                meta['scaler'] = None

            wide_trans[s] = s_data

        self.wide_trans = wide_trans.copy()

        # 4. Construct Cross-Series Lag Features ({series_id}_lag_{k}) across ALL series
        wide_features = pd.DataFrame(index=wide_trans.index)
        for s in series_ids:
            s_lags = normalized_lags[s]
            for lag in s_lags:
                wide_features[f"{s}_lag_{lag}"] = wide_trans[s].shift(lag)

            s_lag_tf = self._get_per_series_param(self.lag_transform, s, None)
            if s_lag_tf is not None:
                for func in s_lag_tf:
                    fname = getattr(func, '__name__', func.__class__.__name__)
                    col_name = f"{s}_{fname}"
                    if hasattr(func, 'shift'):
                        col_name += f"_shift_{func.shift}"
                    if hasattr(func, 'window_size'):
                        col_name += f"_{func.window_size}"
                    if hasattr(func, 'quantile'):
                        col_name += f"_q{func.quantile}"
                    wide_features[col_name] = func(wide_trans[s])

        self.feature_cols = wide_features.columns.tolist()

        # 5. Stack wide feature matrix into panel training dataset (X, y) preserving time index
        X_list = []
        y_list = []

        for i, s in enumerate(series_ids):
            df_s = wide_features.copy()

            # Attach exogenous feature columns if present
            if len(exog_cols) > 0:
                s_dfc = dfc[dfc[self.id_col] == s]
                for c in exog_cols:
                    df_s[c] = s_dfc[c]

            # Encode series identifier column (dummy one-hot, ordinal integer, or native categorical)
            if self.series_encoding == 'dummy':
                for s_other in series_ids:
                    df_s[f"{self.id_col}_{s_other}"] = 1.0 if s_other == s else 0.0
            elif self.series_encoding == 'ordinal':
                df_s[self.id_col] = i
            elif self.series_encoding is None:
                df_s[self.id_col] = pd.Categorical([s] * len(df_s), dtype=self.cat_type)

            y_s = wide_trans[s]
            X_list.append(df_s)
            y_list.append(y_s)

        X_all = pd.concat(X_list, axis=0)
        y_all = pd.concat(y_list, axis=0)

        # Drop initial rows containing lag NaNs
        valid_mask = ~(X_all.isna().any(axis=1) | y_all.isna())
        X_clean = X_all[valid_mask].copy()
        y_clean = y_all[valid_mask].copy()

        return X_clean, y_clean, wide_features

    def fit(self, df: pd.DataFrame) -> None:
        """
        Fit the forecaster to the training DataFrame.
        """
        self.df_train = df.copy()
        X, y, _ = self.data_prep(df)
        self.X = X
        self.y = y
        self.X_cols = X.columns.tolist()

        # Configure categorical feature parameters for LightGBM / CatBoost if applicable
        fit_kwargs = {}
        if self.series_encoding is None and self.model_name in ["LGBMRegressor", "CatBoostRegressor"]:
            cat_cols = [self.id_col] + (self.cat_variables or [])
            if self.model_name == "LGBMRegressor":
                fit_kwargs = {"categorical_feature": cat_cols}
            elif self.model_name == "CatBoostRegressor":
                fit_kwargs = {"cat_features": cat_cols, "verbose": False}

        self.model_fit = self.model.fit(X, y, **fit_kwargs)

    def predict_in_sample(self) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
        """
        Generate in-sample predictions and residuals for all series in original scale.
        """
        if not hasattr(self, "model_fit"):
            raise ValueError("Model has not been fitted yet. Call .fit() before predict_in_sample().")
        
        y_preds_raw = self.model_fit.predict(self.X)
        df_eval = self.X.copy()
        df_eval['_y_raw'] = self.y
        df_eval['_y_pred_raw'] = y_preds_raw

        self.fitted_values = {}
        self.in_samp_resids = {}

        for i, s in enumerate(self.series_ids):
            meta = self.transform_meta[s]
            if self.series_encoding == 'dummy':
                mask = df_eval[f"{self.id_col}_{s}"] == 1.0
            elif self.series_encoding == 'ordinal':
                mask = df_eval[self.id_col] == i
            else:
                mask = df_eval[self.id_col] == s

            sub_preds = df_eval.loc[mask, '_y_pred_raw'].values
            sub_y = df_eval.loc[mask, '_y_raw'].values

            # Inverse scaling if a target scaler was applied
            if meta.get('scaler', None) is not None:
                sub_preds = meta['scaler'].inverse_transform(sub_preds.reshape(-1, 1)).ravel()
                sub_y = meta['scaler'].inverse_transform(sub_y.reshape(-1, 1)).ravel()

            sub_resids = sub_y - sub_preds
            fit_len = len(sub_resids)

            orig_target_s = meta['orig_series'].values[-fit_len:]

            if not meta.get('box_cox', False):
                # When no Box-Cox is used, the residual on original scale is exact
                fitted_s = orig_target_s - sub_resids
                resid_s = sub_resids
            else:
                # If Box-Cox was applied, invert in Box-Cox scale
                trans_s = meta.get('orig_before_boxcox', meta['orig_series']).values[-fit_len:]
                bc_fitted = trans_s - sub_resids
                fitted_s = back_box_cox_transform(
                    y_pred=bc_fitted,
                    lmda=meta['lmda'],
                    shift=meta['is_zero'],
                    box_cox_biasadj=meta.get('biasadj', False)
                )
                resid_s = orig_target_s - fitted_s

            fitted_s = np.nan_to_num(fitted_s, nan=0.0, posinf=0.0, neginf=0.0)
            fitted_s = np.clip(fitted_s, a_min=0.0, a_max=None)

            self.fitted_values[s] = fitted_s
            self.in_samp_resids[s] = resid_s

        return self.fitted_values, self.in_samp_resids

    def _k(self) -> int:
        base = self.X.shape[1]
        if isinstance(self.model, (LinearRegression, Ridge, Lasso, ElasticNet)):
            return base + 2
        return base + 1

    def _ic_base(self):
        k = self._k()
        n = len(self.y)
        rss = np.sum((self.y.to_numpy() - self.model_fit.predict(self.X)) ** 2)
        return k, n, rss

    @property
    def aic(self) -> float:
        k, n, rss = self._ic_base()
        return n * np.log(rss / n) + 2 * k

    @property
    def aicc(self) -> float:
        k, n, rss = self._ic_base()
        aic = n * np.log(rss / n) + 2 * k
        return aic + (2 * k * (k + 1)) / (n - k - 1)

    @property
    def bic(self) -> float:
        k, n, rss = self._ic_base()
        return n * np.log(rss / n) + k * np.log(n)

    @property
    def hqc(self) -> float:
        k, n, rss = self._ic_base()
        return n * np.log(rss / n) + 2 * k * np.log(np.log(n))

    def copy(self):
        return copy.deepcopy(self)

    def forecast(self, H: int, exog: Optional[pd.DataFrame] = None,
                 transform_back: bool = True) -> Dict[str, np.ndarray]:
        """
        Recursive multi-step forecast across all target series.

        Parameters
        ----------
        H : int
            Forecast horizon.
        exog : pd.DataFrame, optional
            Optional exogenous features DataFrame for the future forecast horizon.
        transform_back : bool, default True
            Whether to transform the forecast back to the original scale. This argument is for optuna hyperparameter tuning and can be set to False to optimize forecast accuracy in the transformed scale.
        Returns
        -------
        Dict[str, np.ndarray]
            Dictionary keyed by target series ID with length-H forecast arrays in original scale (non-negative, no NaNs).
        """
        if not hasattr(self, "model_fit"):
            raise ValueError("Model has not been fitted yet. Call .fit() before .forecast().")

        series_ids = self.series_ids
        wide_history = self.wide_trans.copy()

        # 1. Preprocess future exogenous variables if provided
        exog_prepared = None
        if exog is not None:
            exog_c = exog.copy()
            if self.cat_variables is not None:
                exog_c = self.create_encoded_features(exog_c)
            exog_prepared = exog_c

        # 2. Build robust future time index for horizon H (supporting any frequency: daily, hourly, monthly, quarterly, etc.)
        last_idx = wide_history.index[-1]
        if isinstance(last_idx, pd.Timestamp) or isinstance(wide_history.index, pd.DatetimeIndex):
            freq = pd.infer_freq(wide_history.index)
            if freq is None and hasattr(wide_history.index, 'inferred_freq'):
                freq = wide_history.index.inferred_freq
            
            if freq is not None:
                future_indices = pd.date_range(start=last_idx, periods=H + 1, freq=freq)[1:].tolist()
            else:
                dt_diff = pd.Series(wide_history.index).diff().median()
                future_indices = [last_idx + ((i + 1) * dt_diff) for i in range(H)]
        else:
            future_indices = [last_idx + i for i in range(1, H + 1)]

        # Append future horizon placeholders (initialized to NaN) to wide history matrix
        for fut_idx in future_indices:
            wide_history.loc[fut_idx] = np.nan

        # 3. Recursive multi-step forecast loop (step h = 1..H)
        for step_idx in range(H):
            current_time = future_indices[step_idx]
            hist_sub = wide_history.loc[:current_time].copy()

            # Extract cross-series lag feature values at step h across all series
            step_features = {}
            for s in series_ids:
                s_lags = self.normalized_lags[s]
                for lag in s_lags:
                    step_features[f"{s}_lag_{lag}"] = hist_sub[s].iloc[-(lag + 1)]

                s_lag_tf = self._get_per_series_param(self.lag_transform, s, None)
                if s_lag_tf is not None:
                    for func in s_lag_tf:
                        fname = getattr(func, '__name__', func.__class__.__name__)
                        col_name = f"{s}_{fname}"
                        if hasattr(func, 'shift'):
                            col_name += f"_shift_{func.shift}"
                        if hasattr(func, 'window_size'):
                            col_name += f"_{func.window_size}"
                        if hasattr(func, 'quantile'):
                            col_name += f"_q{func.quantile}"
                        tf_series = func(hist_sub[s])
                        step_features[col_name] = tf_series.iloc[-1]

            # Build batch feature matrix rows for all series at step h
            step_rows = []
            for i, s in enumerate(series_ids):
                row = step_features.copy()

                # Inject exogenous features for step h
                if exog_prepared is not None and len(self.exog_cols) > 0:
                    if self.id_col in exog_prepared.columns:
                        s_exog = exog_prepared[(exog_prepared[self.id_col] == s) & (exog_prepared.index == current_time)]
                        if len(s_exog) == 0:
                            s_exog = exog_prepared[exog_prepared[self.id_col] == s].iloc[[step_idx]]
                        for c in self.exog_cols:
                            row[c] = s_exog[c].values[0]
                    else:
                        s_exog = exog_prepared.loc[[current_time]] if current_time in exog_prepared.index else exog_prepared.iloc[[step_idx]]
                        for c in self.exog_cols:
                            row[c] = s_exog[c].values[0]

                # Encode series identifier column
                if self.series_encoding == 'dummy':
                    for s_other in series_ids:
                        row[f"{self.id_col}_{s_other}"] = 1.0 if s_other == s else 0.0
                elif self.series_encoding == 'ordinal':
                    row[self.id_col] = i
                elif self.series_encoding is None:
                    row[self.id_col] = s

                step_rows.append(row)

            X_step = pd.DataFrame(step_rows)[self.X_cols]
            if self.series_encoding is None:
                X_step[self.id_col] = pd.Categorical(X_step[self.id_col], dtype=self.cat_type)

            if self.cat_variables is not None and self.cat_encoder is None:
                for c in self.cat_variables:
                    if c in X_step.columns:
                        X_step[c] = pd.Categorical(X_step[c], dtype=self.cat_dtypes[c])

            # Predict step h across all series
            preds_step = self.model_fit.predict(X_step)

            # Update wide history matrix with step h predictions for recursive future steps
            for i, s in enumerate(series_ids):
                wide_history.loc[current_time, s] = preds_step[i]

        # 4. Post-process predictions: Invert all transformations in exact reverse order
        forecasts = {}
        for s in series_ids:
            meta = self.transform_meta[s]
            s_raw_preds = wide_history.loc[future_indices, s].values.copy()
            pred_series = s_raw_preds

            # Inverse Step 1: Invert Target Scaling FIRST (before undifferencing)
            if meta['scaler'] is not None and transform_back: # Only inverse scale if transform_back is True
                pred_series = meta['scaler'].inverse_transform(pred_series.reshape(-1, 1)).ravel()

            # Inverse Step 2: Invert Seasonal Differencing
            if meta['seasonal_diff'] is not None:
                pred_series = invert_seasonal_diff(meta['orig_before_sdiff'], pred_series, meta['seasonal_diff'])

            # Inverse Step 3: Invert Ordinary Differencing
            if meta['difference'] is not None:
                pred_series = undiff_ts(meta['orig_before_diff'], pred_series, n=meta['difference'])

            # Inverse Step 4: Add Trend Back
            if meta['trend_type'] is not None:
                if meta['trend_type'] == 'linear':
                    trend_fc = forecast_trend(
                        model=meta['lr_model'], H=H,
                        degree=meta['pol_degree'], breakpoints=meta['cps'],
                        n_train=len(meta['orig_series'])
                    )
                elif meta['trend_type'] == 'ets':
                    trend_fc = meta['ets_model_fit'].forecast(H).values
                pred_series = pred_series + trend_fc

            # Inverse Step 5: Invert Box-Cox with domain guarding against NaN
            if meta['box_cox']:
                lmda = meta['lmda']
                if lmda is not None and lmda != 0:
                    min_val = -1.0 / lmda + 1e-6 if lmda > 0 else -1e6
                    pred_series = np.maximum(pred_series, min_val)
                pred_series = back_box_cox_transform(
                    y_pred=pred_series, lmda=meta['lmda'],
                    shift=meta['is_zero'], box_cox_biasadj=meta['biasadj']
                )

            # Inverse Step 6: Convert any NaNs/negatives into non-negative 0.0 forecasts
            pred_series = np.nan_to_num(pred_series, nan=0.0, posinf=0.0, neginf=0.0)
            pred_series = np.clip(pred_series, a_min=0.0, a_max=None)

            forecasts[s] = np.array(pred_series)

        return forecasts

    def cross_validate(
        self,
        df: pd.DataFrame,
        cv_split: int,
        test_size: int,
        metrics: List[Callable],
        step_size: int = 1,
        ref_series_id: Optional[str] = None
    ) -> pd.DataFrame:
        """
        Run time-series cross-validation across all interdependent series in long-format panel data.

        Parameters
        ----------
        df : pd.DataFrame
            Long-format panel DataFrame with time index, series identifier column (id_col), and target column.
        cv_split : int
            Number of cross-validation splits.
        test_size : int
            Number of time steps in each forecast evaluation test set.
        metrics : list of callable
            Metric functions (e.g. [MAE, RMSE, MAPE, MASE]) used to evaluate forecast accuracy.
        step_size : int, default 1
            Step size to move the test window forward in each split fold.
        ref_series_id : str, optional
            Series ID used as reference for time index splitting. If None, the series with the shortest length is used.

        Returns
        -------
        pd.DataFrame
            DataFrame containing detailed predictions (cutoff, fold_index, horizon, split, id_col, y_true, y_pred) for all folds.
            Aggregated metric summary across folds per series is stored in `self.cv_summary` (metrics in index, id_cols in columns).
        """
        dfc = df.copy()
        series_ids = sorted(dfc[self.id_col].unique().tolist())
        
        # 1. Determine reference series for time index splitting (default to shortest series if not specified)
        if ref_series_id is None:
            ref_series_id = min(series_ids, key=lambda s: len(dfc[dfc[self.id_col] == s]))
        
        ref_df = dfc[dfc[self.id_col] == ref_series_id]
        
        tscv = SplitTimeSeries(n_splits=cv_split, test_size=test_size, step_size=step_size)
        
        # Dictionary to accumulate fold scores: {metric_name: {series_id: [fold1_score, fold2_score, ...]}}
        fold_scores = {m.__name__: {s: [] for s in series_ids} for m in metrics}
        cv_rows = []

        for idx, (ref_train_idx, ref_test_idx) in enumerate(tscv.split(ref_df)):
            cutoff_date = ref_df.index[ref_train_idx[-1]]
            test_dates = ref_df.index[ref_test_idx]
            
            # Split long-format panel by date boundaries
            train_fold = dfc[dfc.index <= cutoff_date]
            test_fold = dfc[(dfc.index > cutoff_date) & (dfc.index <= test_dates[-1])]
            
            # Fit model on training fold
            self.fit(train_fold)
            
            # Extract future exogenous variables for test fold
            exog_fold = test_fold.drop(columns=[self.target_col]) if len(self.exog_cols) > 0 else None
            
            # Forecast horizon test_size across all series
            H_fold = len(test_dates)
            fc_dict = self.forecast(H=H_fold, exog=exog_fold)

            # Evaluate metrics and collect forecast results for each series
            for s in series_ids:
                s_test_df = test_fold[test_fold[self.id_col] == s]
                y_true_s = s_test_df[self.target_col].values
                y_pred_s = fc_dict[s][:len(y_true_s)] # Ensure alignment in case of shorter series
                
                s_train_df = train_fold[train_fold[self.id_col] == s]
                y_train_s = s_train_df[self.target_col].values
                
                # Evaluate metrics for series s in this fold
                for m in metrics:
                    mname = m.__name__
                    if mname in ["MASE", "SMAE", "SRMSE", "RMSSE"]:
                        score = m(y_true_s, y_pred_s, y_train_s)
                    else:
                        score = m(y_true_s, y_pred_s)
                    fold_scores[mname][s].append(score)
                
                # Record detailed prediction rows for cv_df_
                for step_h in range(len(y_true_s)):
                    cv_rows.append({
                        "cutoff": cutoff_date,
                        "fold_index": s_test_df.index[step_h],
                        "horizon": step_h + 1,
                        "split": f"fold_{idx + 1}",
                        self.id_col: s,
                        "y_true": y_true_s[step_h],
                        "y_pred": y_pred_s[step_h]
                    })

        cv_df_ = pd.DataFrame(cv_rows)

        # Build self.cv_summary with metrics in index and series_ids (plus overall average) in columns
        summary_dict = {}
        for s in series_ids:
            summary_dict[s] = {m.__name__: np.mean(fold_scores[m.__name__][s]) for m in metrics}
        
        # Add overall average across all series
        summary_dict["overall"] = {m.__name__: np.mean([np.mean(fold_scores[m.__name__][s]) for s in series_ids]) for m in metrics}
        
        summary_df = pd.DataFrame(summary_dict)
        summary_df.index.name = "eval_metric"
        self.cv_summary = summary_df

        return cv_df_

    def copy(self):
        return copy.deepcopy(self)

    def get_name(self):
        return "ml_multi_forecaster"


In [3]:
#| hide
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from lightgbm import LGBMRegressor
from peshbeen.metrics import MAE, RMSE, MAPE, MASE

# # Load sales dataset using peshbeen.datasets
# data_path = 'peshbeen/data/sales_df.csv' if os.path.exists('peshbeen/data/sales_df.csv') else ('../../peshbeen/data/sales_df.csv' if os.path.exists('../../peshbeen/data/sales_df.csv') else '../../../peshbeen/data/sales_df.csv')
# df = pd.read_csv(data_path, parse_dates=['date']).set_index('date')
from peshbeen.datasets import load_sales
df = load_sales()

# Add example categorical feature columns: day_of_week and month
df['day_of_week'] = df.index.dayofweek.astype(str)
df['month'] = df.index.month.astype(str)

# Train / Test split
df_train = df[df.index <= '2025-12-31']
df_test = df[df.index >= '2026-01-01']
df_exog = df_test.drop(columns=['sales']).copy()  # Exogenous features for forecasting
H = len(df_test.index.drop_duplicates())

print("=== Testing ml_multi_forecaster ===")
print(f"Train shape: {df_train.shape}, Test shape: {df_test.shape}, Forecast Horizon H: {H}")

# 1. Test Ridge model with target_scaler=StandardScaler(), cat_variables and OneHotEncoder
print("\n--- Test 1: Ridge model with StandardScaler, cat_variables and OneHotEncoder ---")
forecaster1 = ml_multi_forecaster(
    model=Ridge(),
    id_col='store_item',
    target_col='sales',
    lags=7,
    series_encoding='dummy',
    # difference=1,
    # box_cox=True,
    # target_scaler=StandardScaler(),
    cat_variables=['day_of_week', 'month'],
    categorical_encoder=OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)
forecaster1.fit(df_train)
fc1 = forecaster1.forecast(H=H, exog=df_exog)
assert isinstance(fc1, dict), "Forecast output must be a dictionary"
assert 'store_01_item_01' in fc1, "Key 'store_01_item_01' missing from forecast dict"
assert len(fc1['store_01_item_01']) == H, f"Forecast horizon length must equal H={H}"
for s, f_vals in fc1.items():
    assert not np.any(np.isnan(f_vals)), f"Series {s} contains NaNs!"
    assert np.all(f_vals >= 0.0), f"Series {s} contains negative forecasts!"
print(f"Test 1 Passed! All forecast values are finite and non-negative (>= 0.0) for all 72 series.")

# 2. Test cross_validate()
print("\n--- Test 2: Cross Validation (cross_validate) ---")
cv_df = forecaster1.cross_validate(
    df=df,
    cv_split=3,
    test_size=7,
    metrics=[MAE, RMSE, MASE],
    step_size=7
)
assert isinstance(cv_df, pd.DataFrame), "cross_validate must return a DataFrame"
assert hasattr(forecaster1, "cv_summary"), "forecaster must have cv_summary attribute"
assert 'overall' in forecaster1.cv_summary.columns, "cv_summary must contain 'overall' column"
assert 'MAE' in forecaster1.cv_summary.index, "cv_summary index must contain metric names"
print("Test 2 Passed! cv_df shape:", cv_df.shape, "cv_summary shape:", forecaster1.cv_summary.shape)

# 3. Test exception on invalid series_encoding=None
print("\n--- Test 3: Validation exception for series_encoding=None ---")
try:
    forecaster3 = ml_multi_forecaster(
        model=Ridge(),
        id_col='store_item',
        target_col='sales',
        lags=3,
        series_encoding=None
    )
    raise AssertionError("Should have raised ValueError for series_encoding=None with Ridge")
except ValueError as e:
    print("Test 3 Passed! Caught expected ValueError:", e)

# 4. Test LGBMRegressor with series_encoding=None and native categorical variables
print("\n--- Test 4: LGBMRegressor with series_encoding=None ---")
forecaster4 = ml_multi_forecaster(
    model=LGBMRegressor(verbosity=-1, n_estimators=10),
    id_col='store_item',
    target_col='sales',
    lags=3,
    series_encoding=None,
    cat_variables=['day_of_week', 'month']
)
forecaster4.fit(df_train)
fc4 = forecaster4.forecast(H=H, exog=df_exog)
assert len(fc4['store_01_item_01']) == H
for s, f_vals in fc4.items():
    assert not np.any(np.isnan(f_vals)), f"Series {s} contains NaNs!"
    assert np.all(f_vals >= 0.0), f"Series {s} contains negative forecasts!"
print("Test 4 Passed!")

print("\n=== All tests passed successfully! ===")

=== Testing ml_multi_forecaster ===
Train shape: (131112, 4), Test shape: (2592, 4), Forecast Horizon H: 36

--- Test 1: Ridge model with StandardScaler, cat_variables and OneHotEncoder ---
Test 1 Passed! All forecast values are finite and non-negative (>= 0.0) for all 72 series.

--- Test 2: Cross Validation (cross_validate) ---
Test 2 Passed! cv_df shape: (1512, 7) cv_summary shape: (3, 73)

--- Test 3: Validation exception for series_encoding=None ---
Test 3 Passed! Caught expected ValueError: series_encoding=None is only supported for LGBMRegressor and CatBoostRegressor. Please set series_encoding='dummy' or 'ordinal'.

--- Test 4: LGBMRegressor with series_encoding=None ---
Test 4 Passed!

=== All tests passed successfully! ===
